In [10]:
from langchain.agents import create_agent
from langchain.tools import tool

import os
import requests
from pydantic import BaseModel, Field


os.environ["OPENWEATHER_API_KEY"] = ""

from langchain_groq import ChatGroq

llm = ChatGroq(model="qwen/qwen3-32b")   

# 1. Define the updated schema (Added 'location_name' so the agent knows exactly what it found)
class WeatherOutput(BaseModel):
    location_name: str = Field(description="The formal name of the city/location found by the API")
    temperature: float = Field(description="The current temperature in Celsius")
    wind_speed: float = Field(description="Wind speed in meters/second")
    pressure: int = Field(description="Atmospheric pressure in hPa")
    humidity: int = Field(description="Humidity percentage")
    description: str = Field(description="A brief description of the weather conditions")

# 2. The Agent-Ready Tool Function
@tool
def get_weather(location: str) -> WeatherOutput:
    """
    Fetches comprehensive current weather details for a given city location using the OpenWeather API.
    
    Args:
        location (str): The name of the city. Do NOT use broad country names.
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key or api_key.strip() == "":
        raise ValueError("Error: OPENWEATHER_API_KEY environment variable is empty or not set.")
    
    url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&appid=&units=metric"
    
    params = {
        "q": location,
        "appid": api_key,
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status() 
    data = response.json()
    
    return WeatherOutput(
        location_name=data['name'], # Captures the exact city returned by OpenWeather
        temperature=data['main']['temp'],
        wind_speed=data['wind']['speed'],
        pressure=data['main']['pressure'],
        humidity=data['main']['humidity'],
        description=data['weather'][0]['description']
    )

# 3. Updated System Prompt (Forces the Agent to resolve countries to capital/major cities)
weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        "You are a helpful weather assistant. Use the get_weather tool to look up current weather. "
        "Important: Always pass the specific city requested by the user directly to the tool. "
        "If the user asks for a whole country instead of a city, determine its capital city and "
    ),
)

users_location = input("Enter a city or country to get the current weather: ")
response = weather_agent.invoke({"message":[{"role": "user", "content": users_location}]})
print(response["messages"][-1].content)

The current weather in Paris is as follows:
- **Temperature**: 18.4°C  
- **Wind Speed**: 1.3 m/s  
- **Pressure**: 1025 hPa  
- **Humidity**: 80%  
- **Conditions**: Overcast clouds  

Let me know if you'd like additional details! ☀️
